# 🌦️ Analisi e Processamento di Dati Meteo-Climatici

Benvenuti in questo notebook interattivo! Questa attività *hands-on* è progettata per introdurvi alla gestione pratica dei dati meteorologici e climatici utilizzando Python.

Il focus principale di questa esercitazione sarà l'importazione, la manipolazione e la visualizzazione di dati salvati in due dei formati standard più diffusi nella comunità scientifica:
* **NetCDF (Network Common Data Form):** Un formato auto-descrittivo e indipendente dall'architettura, ideale per array di dati multi-dimensionali (come longitudine, latitudine, tempo, altitudine).
* **GRIB (GRIdded Binary):** Il formato standard dell'Organizzazione Meteorologica Mondiale (WMO) per la trasmissione e l'archiviazione di dati storici e previsionali, pensato per l'ottimizzazione e l'efficienza nello scambio dati operativo.

---

### 🎯 Obiettivi dell'Attività
Durante questa esercitazione esploreremo diversi dataset con scale temporali e finalità molto diverse. Gli obiettivi di questo notebook sono mostrare come:

1. **Gestire e Importare i dati:** scaricare in modo robusto file da archivi esterni (es. Google Drive) e leggerli utilizzando librerie moderne come `xarray`.
2. **Esplorare array multi-dimensionali:** comprendere dimensioni, coordinate e variabili, ed estrarre serie temporali per punti geografici specifici.
3. **Analizzare diverse scale temporali:**
   * **Rianalisi Storiche (ERA5-Land):** esplorare l'andamento orario delle variabili meteorologiche nel passato.
   * **Previsioni a Breve Termine (ICON-2I):** leggere e visualizzare in mappa una previsione meteorologica per le successive 72 ore.
   * **Previsioni Stagionali (SEAS5 ECMWF):** leggere e visualizzare dati previsionali e gestire l'incertezza e gli scenari probabilistici (*ensemble*) aggregando i dati in boxplot mensili.
   * **Proiezioni Climatiche (CMIP6):** analizzare i trend a lungo termine di precipitazione identificati da un modello climatico in proiezione fino al 2055.
4. **Creare visualizzazioni interattive:** utilizzare `plotly` per generare grafici esplorabili (serie temporali con slider, boxplot e mappe).

---

### 🛠️ Prerequisiti e Librerie
Il notebook farà uso di alcune librerie fondamentali per l'analisi dei dati geospaziali e statistici in Python. Eseguiremo le analisi appoggiandoci principalmente a:
* `xarray` e `pandas`: per la manipolazione di array etichettati, estrazioni geografiche e aggregazioni temporali.
* `netCDF4`: engine per la lettura e scrittura dei file `.nc`.
* `cfgrib` (ed `eccodes`): engine per la decodifica dei file `.grib`.
* `plotly`: per la generazione dei grafici interattivi e delle mappe.

> **Nota per Google Colab:** Alcune librerie e dipendenze di sistema (come `cfgrib` e `eccodes`) non sono preinstallate su Colab. Nella cella di codice successiva troverai i comandi necessari per configurare il tuo ambiente di lavoro in pochi secondi.

---

### 🚀 Istruzioni
Per procedere, esegui le celle di codice in sequenza cliccando sul pulsante **Play** `[▶]` a sinistra di ogni blocco, oppure utilizzando la scorciatoia da tastiera `Shift + Enter`.

Buon lavoro e buona esplorazione dei dati!

### 📥 Importazione dei Dati da Google Drive

Prima di poter analizzare i dati meteo-climatici, dobbiamo renderli accessibili al nostro ambiente di lavoro. Poiché le macchine virtuali di Google Colab sono temporanee e si "azzerano" alla fine di ogni sessione, è necessario scaricare i file di partenza ogni volta che ci si connette.

In questa fase trasferiremo i dataset necessari (in formato NetCDF e GRIB) da Google Drive alla memoria locale della nostra sessione Colab. Questo passaggio vi permetterà di:
* **Automatizzare il processo:** evitare di dover caricare manualmente file pesanti dal vostro computer.
* **Lavorare in velocità:** i file salvati nella memoria locale di Colab (`/content/`) vengono letti dalle librerie Python in modo quasi istantaneo.
* **Verificare l'integrità:** controllare che i dati siano stati scaricati completamente e correttamente prima di processarli.

Esegui la cella seguente per avviare il download dei file di esercitazione. Il processo richiederà alcuni secondi. Al termine, potrai verificare visivamente la presenza dei file cliccando sull'icona a forma di cartella, nella barra laterale sinistra del notebook.

In [1]:

# Installazione delle librerie aggiuntive necessarie

!pip install cfgrib cftime xarray netcdf4 plotly matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 5.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 6.5 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 12.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 17.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 21.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 26.2 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 32.9 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 40.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 39.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 40.6 MB/s  0:00:00
   ━━━━━━

In [2]:
import os

# ID dei files di Google Drive da scaricare e salvare nella sessione di Colab
demo_files = {
  "1taxpTGx9cOufdKjeOlm1Sl-Lfw6MBhY5": "reanalysis.nc",
  "1m03jGf0fnAoVbLkIZMesA2nXAcP_5dyd": "seasonal.nc",
  "1t3o2ZuIXOWhuAJ83s_gV2wU6DQaek6DQ": "short_term.grib",
  "1EV0LfT22Z6ac4F0E5lU1Kfq3OGdXs_O0": "projection.nc"
}

# Ciclo sui files
for gid, file_name in demo_files.items():

  print(f'File: "{file_name}"')
  download_url = f'https://docs.google.com/uc?export=download&id={gid}'

  # Scarica il file direttamente nella sessione di Colab
  if not os.path.exists(file_name):
      print("\nScaricamento del file in corso...\n")
      try:
        !wget --no-check-certificate '{download_url}' -O {file_name}
        print("\nDataset già presente. Download non necessario.\n")
      except Exception as e:
        print(f"\nErrore nello scaricamento del file: {e}")
  else:
      print("\nDataset già presente. Download non necessario.\n")

File: "reanalysis.nc"

Scaricamento del file in corso...

--2026-05-28 07:58:55--  https://docs.google.com/uc?export=download&id=1taxpTGx9cOufdKjeOlm1Sl-Lfw6MBhY5
Resolving docs.google.com (docs.google.com)... 142.251.13.101, 142.251.13.113, 142.251.13.100, ...
Connecting to docs.google.com (docs.google.com)|142.251.13.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1taxpTGx9cOufdKjeOlm1Sl-Lfw6MBhY5&export=download [following]
--2026-05-28 07:58:55--  https://drive.usercontent.google.com/download?id=1taxpTGx9cOufdKjeOlm1Sl-Lfw6MBhY5&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.110.132, 2a00:1450:4001:c1f::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.110.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 259111 (253K) [application/octet-stream]
Saving to: ‘reanalysis.nc’

reanalysis.

### 🔍 Esplorazione della Struttura dei Dataset

Prima di procedere con l'analisi e i grafici, è fondamentale comprendere come sono organizzate internamente le informazioni. Stampando a schermo il dataset, possiamo ispezionare facilmente la sua architettura multi-dimensionale.

L'output interattivo generato dalla libreria `xarray` vi permetterà di esplorare quattro sezioni chiave (cliccando sui triangolini a sinistra per espanderle):
* **Dimensioni (Dimensions):** la forma geometrica dell'array (es. il numero totale di ore nel dataset o di punti griglia).
* **Coordinate (Coordinates):** i valori di riferimento degli assi, come la linea temporale esatta (`valid_time`) o i riferimenti spaziali (`latitude`, `longitude`).
* **Variabili (Data variables):** le grandezze fisiche effettive contenute nel file (es. `t2m` per la temperatura a 2 metri).
* **Attributi (Attributes):** i metadati essenziali forniti dal centro climatico (es. ECMWF), che includono convenzioni, unità di misura e storia del file.

Esegui la cella seguente per visualizzare la panoramica completa del dataset ERA5-Land caricato in memoria. Clicca sulle icone a destra di ogni riga per visualizzare gli attributi e i valori interni.

In [3]:

import xarray as xr

# Lettura file in formato NetCDF
ds_reanalysis = xr.open_dataset("reanalysis.nc")
ds_seasonal = xr.open_dataset("seasonal.nc")
ds_projection = xr.open_dataset("projection.nc")

# Lettura file in formato GRIB
ds_short_term = xr.open_dataset("short_term.grib", engine="cfgrib")


In [4]:
print("Dato di rianalisi di temperatura dell'aria a 2 metri di altezza e di precipitazione (dataset: ERA5-Land)")
ds_reanalysis

Dato di rianalisi di temperatura dell'aria a 2 metri di altezza e di precipitazione (dataset: ERA5-Land)


<xarray.Dataset> Size: 478kB
Dimensions:     (valid_time: 720, latitude: 8, longitude: 10)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 6kB 2026-04-01 ... 2026-04-30T23:...
  * latitude    (latitude) float64 64B 40.5 40.4 40.3 40.2 40.1 40.0 39.9 39.8
  * longitude   (longitude) float64 80B 15.7 15.8 15.9 16.0 ... 16.4 16.5 16.6
    expver      (valid_time) <U4 12kB ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 230kB ...
    tp          (valid_time, latitude, longitude) float32 230kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-27T13:55 GRIB to CDM+CF via cfgrib-0.9.1...

In [5]:
print("Previsioni a breve termine di precipitazione (dataset: ICON-2I)")
ds_short_term

Previsioni a breve termine di precipitazione (dataset: ICON-2I)


<xarray.Dataset> Size: 169MB
Dimensions:     (step: 73, latitude: 761, longitude: 761)
Coordinates:
    time        datetime64[ns] 8B ...
  * step        (step) timedelta64[ns] 584B 00:00:00 ... 3 days 00:00:00
    surface     float64 8B ...
  * latitude    (latitude) float64 6kB 33.7 33.72 33.74 ... 48.86 48.88 48.9
  * longitude   (longitude) float64 6kB 3.0 3.025 3.05 ... 21.95 21.97 22.0
    valid_time  (step) datetime64[ns] 584B ...
Data variables:
    tp          (step, latitude, longitude) float32 169MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             cnmc
    GRIB_centreDescription:  Rome
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Rome
    history:                 2026-05-28T07:59 GRIB to CDM+CF via cfgrib-0.9.1...

In [6]:
print("Previsioni stagionali di temperatura e precipitazione (dataset: SEAS5 ECMWF)")
ds_seasonal

Previsioni stagionali di temperatura e precipitazione (dataset: SEAS5 ECMWF)


<xarray.Dataset> Size: 707kB
Dimensions:    (time: 216, number: 51, latitude: 2, longitude: 2)
Coordinates:
  * number     (number) int64 408B 0 1 2 3 4 5 6 7 8 ... 43 44 45 46 47 48 49 50
  * latitude   (latitude) float64 16B 40.5 39.5
  * longitude  (longitude) float64 16B 15.5 16.5
  * time       (time) datetime64[ns] 2kB 2026-03-01 2026-03-02 ... 2026-10-02
Data variables:
    t2m        (time, number, latitude, longitude) float32 176kB ...
    mx2t24     (time, number, latitude, longitude) float32 176kB ...
    mn2t24     (time, number, latitude, longitude) float32 176kB ...
    tp         (time, number, latitude, longitude) float32 176kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-16T07:35 GRIB to CDM+CF via cfgrib-0.9.1...

In [7]:
print("Proiezioni climatiche di precipitazione (dataset: CMIP6, modello BCC-CSM2-MR, scenario SSP5-8.5)")
ds_projection

Proiezioni climatiche di precipitazione (dataset: CMIP6, modello BCC-CSM2-MR, scenario SSP5-8.5)


<xarray.Dataset> Size: 13MB
Dimensions:    (time: 14965, bnds: 2, lat: 10, lon: 12)
Coordinates:
  * time       (time) object 120kB 2015-01-01 12:00:00 ... 2055-12-31 12:00:00
  * lat        (lat) float64 80B 36.45 37.57 38.69 39.81 ... 44.3 45.42 46.54
  * lon        (lon) float64 96B 6.75 7.875 9.0 10.12 ... 15.75 16.88 18.0 19.12
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) object 239kB ...
    lat_bnds   (time, lat, bnds) float64 2MB ...
    lon_bnds   (time, lon, bnds) float64 3MB ...
    pr         (time, lat, lon) float32 7MB ...
Attributes: (12/49)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ScenarioMIP
    branch_method:          Standard
    branch_time_in_child:   0.0
    branch_time_in_parent:  2015.0
    comment:                This is an extension of historical simulation (r1...
    ...                     ...
    title:                  BCC-CSM2-MR output prepared for CMIP6
    tracking_id:            hdl:21.14100/7cd000af-c706-49d2-a4f0-c203a9244275
    variable_id:            pr
    variant_label:          r1i1p1f1
    license:                CMIP6 model data produced by BCC is licensed unde...
    cmor_version:           3.3.2

### ⏪ Dati di Rianalisi Climatica: Visualizzazione Interattiva di una Serie Temporale di Temperatura

I dataset di rianalisi, come **ERA5-Land**, forniscono una ricostruzione storica e coerente delle condizioni meteo-climatiche del passato, combinando osservazioni reali e modelli fisico-matematici. In questa sezione analizzeremo un dataset a risoluzione temporale oraria riferito al mese di aprile 2026.

Poiché il file conserva una struttura spaziale tridimensionale (composta da una piccola griglia di 8 punti di latitudine e 10 di longitudine), il codice applicherà un filtro preventivo per isolare una specifica colonna d'aria. In particolare andremo a:
* **Selezionare il Punto Geografico:** isolando le coordinate di riferimento tramite il metodo di prossimità (`nearest`), che aggancia in automatico il pixel della griglia più vicino alla posizione scelta.
* **Convertire l'Unità di Misura:** passando dai gradi Kelvin originari ai gradi Celsius (°C) mediante la sottrazione della costante $273.15$.
* **Scomporre il MultiIndex:** ripulendo la struttura dell'indice di Pandas dalle coordinate spaziali residue per garantire un vettore temporale lineare e pulito.

Esegui la cella seguente per generare il grafico interattivo. Ti suggerisco di utilizzare lo strumento di zoom di Plotly (cliccando e trascinando con il tasto sinistro del mouse su un'area del grafico) per ingrandire specifiche settimane o giornate: questo ti permetterà di apprezzare chiaramente il ciclo di escursione termica sinusoidale tra il giorno e la notte!

In [14]:
import plotly.graph_objects as go
import pandas as pd
import plotly.io as pio

import plotly.io as pio
pio.renderers.default = "iframe"

# 1. Estrazione dei dati per un singolo punto geografico
# Il metodo 'nearest' aggancia automaticamente il pixel più vicino alla griglia del modello
data_point_t2m = ds_reanalysis.t2m.sel(latitude=40.2, longitude=16.1, method='nearest')

# 2. Conversione e passaggio da gradi Kelvin a Celsius
ts_data_t2m = data_point_t2m.to_series() - 273.15

# 3. Scomposizione del MultiIndex per mantenere esclusivamente il vettore temporale lineare
if isinstance(ts_data_t2m.index, pd.MultiIndex):
    ts_data_t2m = ts_data_t2m.reset_index(level=['latitude', 'longitude'], drop=True)

# 4. Inizializzazione della figura orientata agli oggetti
fig = go.Figure()

# 5. Aggiunta della traccia lineare per i dati termici
fig.add_trace(
    go.Scatter(
        x=ts_data_t2m.index,
        y=ts_data_t2m.values,
        mode='lines',
        name='Temp a 2m',
        line=dict(color='#ef553b', width=1.5),
        hovertemplate='Data e Ora: %{x|%d/%m/%Y %H:%M}<br>Temperatura: %{y:.2f} °C<extra></extra>'
    )
)

# 6. Formattazione e personalizzazione del layout del grafico
fig.update_layout(
    title=dict(
        text="Andamento Temporale della Temperatura Oraria a 2 Metri (ERA5-Land)",
        font=dict(size=18, family="Arial", color="#333333"),
        x=0.5
    ),
    xaxis=dict(
        title="Data e Ora",
        showgrid=True,
        gridcolor='#e5e5e5',
        type="date",
        tickformat="%d/%m/%Y"
    ),
    yaxis=dict(
        title="Temperatura (°C)",
        showgrid=True,
        gridcolor='#e5e5e5',
        zeroline=False
    ),
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=50, r=30, t=70, b=40)
)

# 7. Rendering del grafico interattivo
fig.show()

### ⏪ Dati di Rianalisi Climatica: Visualizzazione Interattiva di una Serie Temporale di Precipitazione

La gestione delle precipitazioni all'interno dei dataset storici (come ERA5-Land) richiede spesso un passaggio di calcolo aggiuntivo rispetto alla temperatura. Molti centri di calcolo meteorologici, infatti, non memorizzano la pioggia caduta singolarmente in quell'ora, ma registrano un **accumulo progressivo che cresce durante la giornata e si azzera automaticamente a mezzanotte**.

Se provassimo a graficare direttamente il dato grezzo, otterremmo un andamento artificiale "a denti di sega", con linee che salgono costantemente per poi crollare bruscamente a zero ogni notte.

Per ottenere la **reale precipitazione oraria**, in questa sezione applicheremo un algoritmo di **decumulo condizionale**:
1. **Differenziazione Temporale:** calcoliamo la differenza tra l'ora corrente e l'ora precedente (`.diff()`) per isolare la pioggia caduta esclusivamente in quel segmento di 60 minuti.
2. **Gestione del Reset Notturno:** intercettiamo il momento in cui il contatore del modello si azzera (alle ore 01:00 di ogni nuovo giorno), ripristinando il valore corretto per evitare picchi negativi nel grafico.
3. **Pulizia Numerica:** eliminiamo eventuali micro-errori di arrotondamento decimale tramite una funzione di filtraggio del limite inferiore (`clip`).

Esegui la cella seguente per elaborare il dataset e visualizzare il grafico finale in millimetri all'ora (mm/h). Anche in questo caso, la mole di dati orari è imponente (oltre 8700 punti per un anno intero): utilizza lo zoom del mouse per isolare i singoli eventi alluvionali o i temporali estivi di tuo interesse!

In [15]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# 1. Estrazione dei dati per un singolo punto geografico
# Moltiplicazione per 1000 per convertire l'unità nativa in metri nell'equivalente in millimetri (mm)
data_point_tp = ds_reanalysis.tp.sel(latitude=40.2, longitude=16.1, method='nearest') * 1000

# 2. Algoritmo di decumulo orario per la gestione del reset del contatore (ore 01:00)
da_differenza = data_point_tp.diff(dim='valid_time')
da_decumulato = data_point_tp.copy(deep=True)
da_decumulato.values[1:] = da_differenza.values

is_ora_uno = data_point_tp['valid_time'].dt.hour == 1
da_decumulato = da_decumulato.where(~is_ora_uno, data_point_tp)
da_decumulato = da_decumulato.clip(min=0)

# 3. Conversione e scomposizione del MultiIndex per isolare l'asse temporale lineare
ts_data_tp = da_decumulato.to_series()
if isinstance(ts_data_tp.index, pd.MultiIndex):
    ts_data_tp = ts_data_tp.reset_index(level=['latitude', 'longitude'], drop=True)

# 4. Inizializzazione della figura orientata agli oggetti
fig = go.Figure()

# 5. Aggiunta della traccia con riempimento ad area per i dati pluviometrici
fig.add_trace(
    go.Scatter(
        x=ts_data_tp.index,
        y=ts_data_tp.values,
        mode='lines',
        name='Pioggia Oraria',
        line=dict(color='#3498db', width=1.2),
        fill='tozeroy',
        fillcolor='rgba(52, 152, 219, 0.25)',
        hovertemplate='Data e Ora: %{x|%d/%m/%Y %H:%M}<br>Precipitazione Oraria: %{y:.2f} mm<extra></extra>'
    )
)

# 6. Formattazione e personalizzazione del layout del grafico
fig.update_layout(
    title=dict(
        text="Andamento Temporale della Precipitazione Oraria (ERA5-Land)",
        font=dict(size=18, family="Arial", color="#333333"),
        x=0.5
    ),
    xaxis=dict(
        title="Data e Ora",
        showgrid=True,
        gridcolor='#e5e5e5',
        type="date",
        tickformat="%d/%m/%Y"
    ),
    yaxis=dict(
        title="Precipitazione Oraria [mm]",
        showgrid=True,
        gridcolor='#e5e5e5',
        zeroline=True,
        zerolinecolor='#cccccc'
    ),
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=50, r=30, t=70, b=40)
)

# 7. Rendering del grafico interattivo
fig.show()

### 🗺 Previsioni Meteorologiche a Breve Termine: Visualizzazione Interattiva di una Mappa

Finora abbiamo analizzato i dati meteo-climatici estraendo singole serie temporali per un punto specifico (visione *monodimensionale*). Tuttavia, la vera forza dei modelli ad alta risoluzione risiede nella loro capacità di descrivere l'evoluzione dei fenomeni nello spazio.

In questa sezione faremo un salto di qualità: abbandoneremo il singolo punto per **visualizzare l'intera mappa geografica delle precipitazioni** previste dal modello **ICON-2I** sull'Italia, focalizzandoci su un preciso istante nel tempo.

Il codice è stato ottimizzato per offrirvi una mappa interattiva avanzata:
* **Selezione dello Step Temporale:** impostando la variabile `step_scelto` (es. `24`, ovvero a +24 ore dall'inizio della simulazione), potrai isolare qualsiasi momento delle 72 ore di previsione per analizzare dove si troverà la perturbazione.
* **Effetto Campo Grigliato (Continuous Grid):** a differenza dei grafici a punti isolati, questa mappa fonde i dati ad alta risoluzione del modello per creare un campo cromatico continuo. Osserverai le reali sfumature e la struttura spaziale dei fronti piovosi sopra la cartografia.
* **Sfondo Geografico Reale:** i dati meteo sono sovrapposti a una mappa fisica chiara (*Carto-Positron*). Potrai identificare istantaneamente coste, confini regionali e città colpite dalle piogge.
* **Navigazione Fluida e Scroll Zoom:** per facilitare lo studio del territorio, abbiamo sbloccato i comandi nativi di navigazione. Potrai muoverti sulla mappa trascinando con il tasto sinistro e **utilizzare liberamente la rotellina del mouse** (o il *pinch-to-zoom* del trackpad) per ingrandire i dettagli fino a livello locale.

Esegui la cella per generare la mappa grigliata e inizia a esplorare il territorio.

In [16]:
import plotly.express as px
import pandas as pd
import numpy as np

# ==========================================
# 📐 SCEGLI QUI LO STEP TEMPORALE DA VEDERE
# (Orizzonte temporale da 0 a 72 ore)
step_scelto = 24
# ==========================================

# 1. Selezione dello step temporale e campionamento della griglia spaziale
# Estrazione del singolo step e campionamento dei punti (passo 2) per ottimizzare la fluidità
ds_step = ds_short_term.tp.isel(step=step_scelto, latitude=slice(0, None, 2), longitude=slice(0, None, 2))

# Conversione del DataArray bidimensionale in un DataFrame Pandas
df_grid_map = ds_step.to_dataframe().reset_index()

# 2. Creazione della mappa a griglia continua (Density Mapbox)
fig = px.density_mapbox(
    df_grid_map,
    lat='latitude',
    lon='longitude',
    z='tp',
    radius=12,                   # Raggio di fusione dei pixel per ottenere un campo continuo
    center=dict(lat=42.0, lon=12.5), # Centratura geografica iniziale sulla penisola italiana
    zoom=5.0,                    # Livello di ingrandimento cartografico iniziale
    mapbox_style="carto-positron", # Configurazione di uno sfondo cartografico chiaro e pulito
    color_continuous_scale="Blues", # Scala cromatica sequenziale associata ai millimetri di pioggia
    range_color=[0, df_grid_map['tp'].max()], # Calibrazione dinamica dei limiti della scala colore
    labels={'tp': '[mm]'},
    title=f"Precipitazione Cumulata - Previsione a +{step_scelto} ore (ICON-2I)"
)

# 3. Formattazione e personalizzazione del layout del grafico
fig.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    title_font=dict(size=18, family="Arial", color="#333333"),
    width=900,
    height=750
)

# Personalizzazione dei parametri informativi visualizzati al passaggio del mouse
fig.update_traces(
    hovertemplate='<b>Latitudine:</b> %{lat:.2f}°<br><b>Longitudine:</b> %{lon:.2f}°<br><b>Precipitazione:</b> %{z:.2f} mm<extra></extra>'
)

# 4. Rendering del grafico interattivo con abilitazione dello zoom tramite rotella
fig.show(config={'scrollZoom': True})

/tmp/ipykernel_81/557450170.py:19: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(


### 📈 Estrazione di un Membro dall'Ensemble di Previsione Stagionale

Dopo aver analizzato la distribuzione probabilistica mensile dell'intero *ensemble*, potremmo voler scendere nel dettaglio e osservare la traiettoria giornaliera della temperatura per uno specifico scenario futuro.

In questa sezione andremo a tracciare la serie temporale di un singolo membro (nel codice preimpostato, il membro `25`). Il codice è stato strutturato per essere facilmente esplorabile:
* **Selezione dinamica dello scenario:** definendo la variabile `member`, potete cambiare facilmente numero (da 0 a 50 per il dataset SEAS5) per esplorare simulazioni differenti.
* **Localizzazione geografica:** estraiamo i dati per le coordinate (40.0°, 16.2°) sfruttando il metodo di aggancio automatico al punto griglia più vicino (`nearest`).
* **Orizzonte temporale:** limitiamo l'estrazione giornaliera ai primi **180 giorni** (circa 6 mesi), concentrandoci sul periodo di maggiore rilevanza.

Esegui la cella seguente per visualizzare l'evoluzione giornaliera della temperatura prevista da questo specifico scenario. **Prova a modificare il valore di `member = 25`** nella prima riga di codice per scoprire come altri scenari simulano l'andamento meteorologico degli stessi mesi!

In [17]:
import plotly.graph_objects as go
import pandas as pd

# ==========================================
# 📐 SCEGLI QUI IL MEMBRO ENSEMBLE DA VEDERE
# (Indice del membro da 0 a 50)
member = 25
# ==========================================

# 1. Estrazione dei dati per un singolo punto geografico e membro specifico
data_point_member = ds_seasonal.t2m.sel(number=member, latitude=40.0, longitude=16.2, method='nearest')

# 2. Slicing temporale sui primi 180 giorni e passaggio da gradi Kelvin a Celsius
ts_data_member = data_point_member.isel(time=slice(0, 180)).to_series() - 273.15

# 3. Scomposizione del MultiIndex per mantenere esclusivamente il vettore temporale lineare
if isinstance(ts_data_member.index, pd.MultiIndex):
    ts_data_member = ts_data_member.reset_index(level=['latitude', 'longitude'], drop=True)

# 4. Inizializzazione della figura orientata agli oggetti
fig = go.Figure()

# 5. Aggiunta della traccia lineare per i dati termici del singolo scenario
fig.add_trace(
    go.Scatter(
        x=ts_data_member.index,
        y=ts_data_member.values,
        mode='lines',
        name=f'Membro {member}',
        line=dict(color='#ef553b', width=1.5),
        hovertemplate='Data: %{x|%d/%m/%Y}<br>Temperatura: %{y:.2f} °C<extra></extra>'
    )
)

# 6. Formattazione e personalizzazione del layout del grafico
fig.update_layout(
    title=dict(
        text=f"Previsione Stagionale di Temperatura Media Giornaliera a 2 Metri (Membro {member}, SEAS5 ECMWF)",
        font=dict(size=16, family="Arial", color="#333333"),
        x=0.5
    ),
    xaxis=dict(
        title="Data",
        showgrid=True,
        gridcolor='#e5e5e5',
        type="date",
        tickformat="%d/%m/%Y"
    ),
    yaxis=dict(
        title="Temperatura [°C]",
        showgrid=True,
        gridcolor='#e5e5e5',
        zeroline=False
    ),
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=50, r=30, t=70, b=40)
)

# 7. Rendering del grafico interattivo
fig.show()

### 📊 Distribuzione Probabilistica Mensile dell'Ensemble di Previsione Stagionale

Nelle previsioni stagionali, l'obiettivo non è prevedere la temperatura esatta di un giorno specifico, ma comprendere la **tendenza generale del mese** (es. "Sarà un mese mediamente più caldo o più freddo della norma?").

In questo passaggio andremo a:
1. **Selezionare il punto di interesse:** utilizzando le coordinate geografiche (40.0°, 16.2°) tramite il metodo di prossimità (`nearest`), che aggancia in automatico il punto della griglia del modello più vicino.
2. **Aggregare i dati:** calcolando la **media mensile** per ciascuno dei 51 scenari (membri dell'ensemble) disponibili nel dataset.
3. **Filtrare l'orizzonte temporale:** concentrandoci esclusivamente sui primi **6 mesi** di previsione, che rappresentano la finestra temporale più affidabile.

Visualizzeremo infine la distribuzione di questi 51 scenari mese per mese utilizzando dei *Box Plot* (diagrammi a scatola). Questo strumento grafico è ideale per quantificare visivamente l'incertezza:
* **La scatola centrale:** contiene il 50% degli scenari più probabili (tra il 25° e il 75° percentile).
* **La linea orizzontale al suo interno:** rappresenta la mediana (lo scenario "centrale").
* **I baffi e i punti isolati:** indicano rispettivamente la dispersione generale del modello e le eventuali anomalie estreme (*outliers*). Più la scatola risulta "alta", maggiore è l'incertezza della previsione per quel determinato mese.

Esegui la cella seguente per esplorare l'ensemble previsionale della temperatura.

In [18]:
import plotly.graph_objects as go
import pandas as pd

# 1. Estrazione dei dati per un singolo punto geografico e conversione
# Selezione del punto griglia tramite coordinate lineari e passaggio da gradi Kelvin a Celsius
data_point_seasonal = ds_seasonal.t2m.sel(latitude=40.0, longitude=16.2, method='nearest') - 273.15

# 2. Aggregazione temporale e calcolo della media mensile
# Raggruppamento dei dati giornalieri per l'inizio di ciascun mese (1 Month Start) lungo la dimensione 'time'
monthly_data = data_point_seasonal.resample(time='1MS').mean(dim='time')

# 3. Slicing temporale e conversione in DataFrame Pandas
# Limitazione dell'orizzonte previsionale ai primi 6 mesi ed estrazione della struttura dati
monthly_data = monthly_data.isel(time=slice(0, 6))
df_seasonal = monthly_data.to_dataframe().reset_index()

# Mappatura manuale dei mesi in lingua italiana per uniformità di visualizzazione dell'asse X
mesi_it = {1:'Gen', 2:'Feb', 3:'Mar', 4:'Apr', 5:'Mag', 6:'Giu',
           7:'Lug', 8:'Ago', 9:'Set', 10:'Ott', 11:'Nov', 12:'Dic'}

# Generazione di un'etichetta temporale personalizzata combinando mese e anno (es. "Gen 2026")
df_seasonal['Etichetta_Mese'] = df_seasonal['time'].dt.month.map(mesi_it) + " " + df_seasonal['time'].dt.year.astype(str)

# 4. Inizializzazione della figura orientata agli oggetti
fig = go.Figure()

# 5. Aggiunta della traccia statistica Box Plot per i membri dell'ensemble
fig.add_trace(
    go.Box(
        x=df_seasonal['Etichetta_Mese'],
        y=df_seasonal['t2m'],
        name='Ensemble (51 membri)',
        marker_color='red',
        line=dict(width=1.5),
        boxpoints='outliers', # Evidenziazione visiva dei soli scenari estremi isolati
        hovertemplate='Mese: %{x}<br>Temperatura Media: %{y:.2f} °C'
    )
)

# 6. Formattazione e personalizzazione del layout del grafico
fig.update_layout(
    title=dict(
        text="Ensemble Previsionale SEAS5 ECMWF della Temperatura a 2 Metri (+6 mesi)",
        font=dict(size=18, family="Arial", color="#333333"),
        x=0.5
    ),
    xaxis=dict(
        title="Mese",
        showgrid=True,
        gridcolor='#e5e5e5',
        type="category"
    ),
    yaxis=dict(
        title="Temperatura Media Mensile [°C]",
        showgrid=True,
        gridcolor='#e5e5e5',
        zeroline=False
    ),
    template="plotly_white",
    margin=dict(l=50, r=30, t=70, b=40)
)

# 7. Rendering del grafico interattivo
fig.show()

### ⏩ Visualizzazione Riassuntiva di una Proiezione Climatica

Per visualizzare la variabilità climatica in modo immediato e intuitivo, possiamo adattare il concetto delle *Warming Stripes* (le strisce del riscaldamento globale) al tema delle precipitazioni.

In questo grafico non utilizzeremo linee o assi cartesiani, ma rappresenteremo ogni anno come una **striscia verticale colorata**:

Questo strumento visivo permette di identificare a colpo d'occhio l'alternanza di cicli di siccità prolungata o l'intensificazione di anni estremamente piovosi nel corso del prossimo trentennio.

Esegui la cella per generare il codice delle strisce climatiche.

In [19]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# 1. Estrazione dei dati per un singolo punto geografico e conversione
# Moltiplicazione per 86400 per convertire il flusso nativo (kg/m²/s) in millimetri giornalieri (mm/giorno)
ts_daily_mm = ds_projection.pr.sel(lat=40.0, lon=16.2, method='nearest') * 86400

# 2. Aggregazione temporale e strutturazione della matrice bidimensionale
# Somma dei dati giornalieri per ottenere la precipitazione totale cumulata annua
annual_precip = ts_daily_mm.groupby('time.year').sum(dim='time')

# Conversione in DataFrame Pandas e strutturazione del vettore bidimensionale per la Heatmap
df_annual = annual_precip.to_dataframe(name='pioggia_annua').reset_index()
matrice_stripes = [df_annual['pioggia_annua'].values]
anni = df_annual['year'].values

# 3. Inizializzazione della figura orientata agli oggetti
fig = go.Figure()

# 4. Creazione del grafico a strisce tramite Heatmap spaziale
fig.add_trace(
    go.Heatmap(
        x=anni,
        y=["Precipitazione"],
        z=matrice_stripes,
        # Utilizzo della scala divergente 'BrBG' (Brown-Green) per evidenziare i deficit e i surplus idrici
        colorscale='BrBG',
        showscale=True,
        colorbar=dict(title="Precipitazione [mm]"),
        hovertemplate='Anno: %{x}<br>Precipitazione: %{z:.1f} mm<extra></extra>'
    )
)

# 5. Formattazione e personalizzazione del layout per isolare le strisce cromatiche
fig.update_layout(
    title=dict(
        text="Strisce Climatiche della Precipitazione (CMIP6, Modello BCC-CSM2-MR, Periodo 2015-2055)",
        font=dict(size=16, family="Arial", color="#333333"),
        x=0.5
    ),
    xaxis=dict(
        title="Anni",
        tickmode='linear',
        dtick=5,
        showgrid=False
    ),
    yaxis=dict(
        visible=False # Rimozione dell'asse verticale per isolare l'effetto geometrico delle strisce
    ),
    template="plotly_white",
    margin=dict(l=30, r=30, t=70, b=40),
    height=250 # Profilo basso e allungato tipico del design espressivo delle Climate Stripes
)

# 6. Rendering del grafico interattivo
fig.show()